In [0]:
from pyspark.sql.functions import col, sum, coalesce, lit, round

orders = spark.table("ecommerce_project.silver.orders")
order_items = spark.table("ecommerce_project.silver.order_items")
refunds = spark.table("ecommerce_project.silver.order_item_refunds")
products = spark.table("ecommerce_project.silver.products")

In [0]:
#Aggregate refunds in case one item has multiple refunds

refund_summary = (
    refunds.groupBy("order_item_id")
    .agg(sum("refund_amount_usd").alias("refund_amount_usd")
)
)

In [0]:
fact_sales_df = (
    order_items.alias("oi")
    .join(
        orders.alias("o"), 
        col("oi.order_id") == col("o.order_id"),
        "inner")
    .join(
        products.alias("p"), 
        col("oi.product_id") == col("p.product_id"),
        "inner")
    .join(
        refund_summary.alias("r"), 
        col("oi.order_item_id") == col("r.order_item_id"),
        "left")
    .select(
        col("oi.order_item_id"),
        col("oi.order_id"),
        col("o.website_session_id"),
        col("o.user_id"),
        col("oi.product_id"),
        col("p.product_name"),
        col("oi.order_item_date").alias("sales_date"),
        col("oi.is_primary_item"),
        col("oi.price_usd"),
        col("oi.cogs_usd"),
        coalesce(
            col("r.refund_amount_usd"),
            lit(0)
        ).alias("refund_amount_usd")
    )
    .withColumn(
        "net_revenue_usd",
        round(col("price_usd") - col("refund_amount_usd"), 2)
    )
    .withColumn(
        "net_profit_usd",
        round(
            col("price_usd")
            - col("refund_amount_usd")
            - col("cogs_usd"),
            2
        )
        )
    )


In [0]:
# Write Gold table
(
    fact_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project.gold.fact_sales")
)

In [0]:
# Verify count
print("Gold fact_sales row count:",spark.table("ecommerce_project.gold.fact_sales").count())

In [0]:
spark.table("ecommerce_project.gold.fact_sales").printSchema()

In [0]:
display(fact_sales.limit(20))

In [0]:
from pyspark.sql.functions import sum, count, countDistinct, sum, round
#load gold fact table
fact_sales = spark.table("ecommerce_project.gold.fact_sales")

In [0]:
#create daily KPIs

daily_sales_df = (
    fact_sales.groupBy("sales_date")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        count("order_item_id").alias("units_sold"),
        round(sum("price_usd"), 2).alias("gross_revenue_usd"),
        round(sum("cogs_usd"), 2).alias("total_cogs_usd"),
        round(sum("refund_amount_usd"), 2).alias("total_refunds_usd"),
        round(sum("net_revenue_usd"), 2).alias("net_revenue_usd"),
        round(sum("net_profit_usd"), 2).alias("net_profit_usd")

    )
)

In [0]:
#write the table

(
    daily_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ecommerce_project.gold.daily_sales_summary")
)

In [0]:
display(
    spark.table("ecommerce_project.gold.daily_sales_summary")
    .orderBy("sales_date")
)

In [0]:
from pyspark.sql.functions import col, countDistinct, sum, when, coalesce, lit, round

sessions = spark.table("ecommerce_project.silver.website_sessions")

orders = spark.table("ecommerce_project.silver.orders")


In [0]:
#summarize orders per website session
orders_by_session = (
    orders.groupBy("website_session_id")
    .agg(
        countDistinct("order_id").alias("session_orders"),
        sum("price_usd").alias("session_revenue_usd")
    )
)

In [0]:
# Join all sessions with their order information
session_sales_df = (
    sessions
    .join(
        orders_by_session,
        on="website_session_id",
        how="left"
    )
    .fillna({
        "session_orders": 0,
        "session_revenue_usd": 0
    })
)

In [0]:
# Calculate channel-level KPIs
channel_performance_df = (
    session_sales_df
    .groupBy("traffic_channel")
    .agg(
        countDistinct("website_session_id").alias("total_sessions"),
        countDistinct("user_id").alias("total_users"),
        sum("session_orders").alias("total_orders"),
        sum(
            when(col("session_orders") > 0, 1).otherwise(0)
        ).alias("converted_sessions"),
        round(
            sum("session_revenue_usd"), 2
        ).alias("total_revenue_usd")
    )
    .withColumn(
        "conversion_rate_percentage",
        round(
            col("converted_sessions") / col("total_sessions") * 100,
            2
        )
    )
    .withColumn(
        "revenue_per_session_usd",
        round(
            col("total_revenue_usd") / col("total_sessions"),
            2
        )
    )
)

In [0]:
#write the table
(
    channel_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project.gold.channel_performance")
)



In [0]:
display(
    spark.table("ecommerce_project.gold.channel_performance")
)

### Product wise analysis

In [0]:
fact_sales = spark.table("ecommerce_project.gold.fact_sales")

product_performance_df = (
    fact_sales
    .groupBy("product_id", "product_name")
    .agg(
        count("order_item_id").alias("units_sold"),
        countDistinct("order_id").alias("total_orders"),
        sum(
            when(col("refund_amount_usd") > 0, 1).otherwise(0)
        ).alias("refunded_items"),
        round(sum("price_usd"), 2).alias("gross_revenue_usd"),
        round(sum("refund_amount_usd"), 2).alias("total_refunds_usd"),
        round(sum("net_revenue_usd"), 2).alias("net_revenue_usd"),
        round(sum("cogs_usd"), 2).alias("total_cogs_usd"),
        round(sum("net_profit_usd"), 2).alias("net_profit_usd")
    )
    .withColumn(
        "refund_rate_percentage",
        round(
            col("refunded_items") / col("units_sold") * 100, 2
        )
    )
)

In [0]:
(
    product_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project.gold.product_performance")
)

In [0]:
display(
    spark.table("ecommerce_project.gold.product_performance"))

### Creating Gold session-funnel table: 
This table will connect website sessions, pageviews and orders. Help analysing landing pages, bounce rate and conversions

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, count, countDistinct, sum, when, col

sessions = spark.table("ecommerce_project.silver.website_sessions")
orders = spark.table("ecommerce_project.silver.orders")
pageviews = spark.table("ecommerce_project.silver.website_pageviews")

In [0]:
#Finding first page viewed during each session
session_window = Window.partitionBy(
    "website_session_id"
    ).orderBy(
        "created_at",
        "website_pageview_id"
    )

landing_pages_df = (
    pageviews
    .withColumn("page_number", row_number().over(session_window))
    .filter(col("page_number") == 1)
    .select(
    "website_session_id",
    col("pageview_url").alias("landing_page")
    )
        
)


In [0]:
#Calculate pageviews and orders per session
pageview_counts_df = (
    pageviews
    .groupBy("website_session_id")
    .agg(count("website_pageview_id").alias("total_pageviews"))
)

session_orders_df = (
    orders
    .groupBy("website_session_id")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        sum("price_usd").alias("revenue_usd")
    )
)

In [0]:
# create and save the session-funnel table
session_funnel_df = (
    sessions
    .join(pageview_counts_df, "website_session_id", "left")
    .join(landing_pages_df, "website_session_id", "left")
    .join(session_orders_df, "website_session_id", "left")
    .fillna({
        "total_pageviews": 0,
        "total_orders": 0,
        "revenue_usd": 0
    })
    .withColumn(
        "is_bounce",
        when(col("total_pageviews") == 1, 1).otherwise(0)
    )
    .withColumn(
        "is_converted",
        when(col("total_orders") > 0, 1).otherwise(0)
    )
)

(
    session_funnel_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project.gold.session_funnel")
)

print(
    spark.table("ecommerce_project.gold.session_funnel").count()
)

In [0]:
display(spark.table("ecommerce_project.gold.session_funnel"))

In [0]:
from pyspark.sql import functions as F
session_funnel = spark.table(
    "ecommerce_project.gold.session_funnel"
)


#This produces one row per landing page with sessions, bounces, conversions, orders and revenue.
#Using F.function_name() prevents conflicts with Python functions.

landing_page_performance_df = (
    session_funnel
    .fillna({"landing_page": "unknown"})
    .groupBy("landing_page")
    .agg(
        F.countDistinct("website_session_id").alias("total_sessions"),
        F.sum("is_bounce").alias("bounced_sessions"),
        F.sum("is_converted").alias("converted_sessions"),
        F.sum("total_orders").alias("total_orders"),
        F.round(
            F.sum("revenue_usd"), 2
        ).alias("total_revenue_usd")
    )
    .withColumn(
        "bounce_rate_percentage",
        F.round(
            F.col("bounced_sessions")
            / F.col("total_sessions") * 100,
            2
        )
    )
    .withColumn(
        "conversion_rate_percentage",
        F.round(
            F.col("converted_sessions")
            / F.col("total_sessions") * 100,
            2
        )
    )
)

In [0]:
(
    landing_page_performance_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "ecommerce_project.gold.landing_page_performance"
    )
)

display(
    spark.table(
        "ecommerce_project.gold.landing_page_performance"
    ).orderBy(col("total_sessions").desc())
)

In [0]:
#Date dimension table for filtering and reporting by year, quarter, month and weekday.

from pyspark.sql import functions as F

date_range = (
    spark.table("ecommerce_project.silver.orders")
    .agg(
        F.min("order_date").alias("start_date"),
        F.max("order_date").alias("end_date")
    )
)

date_dimension_df = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("start_date"),
                F.col("end_date"),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("date")
    )
    .withColumn(
        "date_key",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month_number", F.month("date"))
    .withColumn("month_name", F.date_format("date", "MMMM"))
    .withColumn("week_number", F.weekofyear("date"))
    .withColumn("day_of_month", F.dayofmonth("date"))
    .withColumn("day_name", F.date_format("date", "EEEE"))
)

In [0]:
(
    date_dimension_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce_project.gold.dim_date")
)


In [0]:
display(
    spark.table("ecommerce_project.gold.dim_date")
    .orderBy("date")
)